# Performance metrics

In [167]:
import pandas as pd 
import matplotlib.pyplot as pyplot
import seaborn as sns
import os

In [168]:
df = pd.read_csv("clean_data/all_data_merged.csv")

# KPI 1 : Completion Rate

**Defintion** : measures the percentage of visits that reach the Confirm step after progressing through all funnel steps from Start, including visits where users navigate back one or more steps before completing the journey.

In [169]:
df_sorted = df.sort_values(['visit_id', 'date_time'])

In [170]:
ordered_steps = df_sorted.groupby('visit_id')['process_step'].apply(list)

In [171]:
required_order = ['start', 'step_1','step_2','step_3','confirm']

#Returns True if the required steps appear in the correct order within the user's sequence. Repeats and loops are allowed between steps.

def is_complete_in_order(steps_sequence, required=required_order):
    required_index = 0
    
    for step in steps_sequence:
        if step == required[required_index]:
            required_index += 1
            if required_index == len(required):
                return True
    
    return False

In [172]:
completed = ordered_steps.apply(is_complete_in_order)

In [173]:
completion_rate = completed.mean()

In [174]:
completion_rate_by_variation = completed.groupby(df_sorted.groupby('visit_id')['variation'].first()).mean()
completion_rate_by_variation

variation
Control    0.458489
Test       0.480106
Name: process_step, dtype: float64

In [175]:
completion_rate_v2 = (completion_rate_by_variation * 100).round(2)
completion_rate_v2

variation
Control    45.85
Test       48.01
Name: process_step, dtype: float64

**Conclusion** : 

# KPI 2 : Time spent per step

In [176]:
df = df.sort_values(['visit_id', 'date_time'])
df.head()

,client_id,client_tenure_yr,client_tenure_month,client_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,variation,visitor_id,visit_id,process_step,date_time
300410,3561384,4.0,56.0,60,Unknown,2,63130.44,6,9,Test,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17
300409,3561384,4.0,56.0,60,Unknown,2,63130.44,6,9,Test,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09
76854,7338123,7.0,88.0,24,Male,2,26436.73,6,9,Test,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56
76853,7338123,7.0,88.0,24,Male,2,26436.73,6,9,Test,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12
76852,7338123,7.0,88.0,24,Male,2,26436.73,6,9,Test,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21


In [177]:
df['next_time'] = df.groupby('visit_id')['date_time'].shift(-1)

In [178]:
df['date_time'] = pd.to_datetime(df['date_time'])
df['next_time'] = pd.to_datetime(df['next_time'])

In [179]:
df['time_spent_clean'] = (df['next_time'] - df['date_time']).dt.total_seconds()

In [180]:
# handeling last step ( confirm) 
df = df[df['time_spent_clean'].notna()]

In [181]:
# fixing repeated steps
step_time_per_visit = (
    df.groupby(['visit_id', 'process_step'])['time_spent_clean']
    .sum()
    .reset_index())

### Handeling time spent outliers

In [182]:
print(df['time_spent_clean'].describe())

count    247940.000000
mean         84.217500
std         216.018401
min           0.000000
25%          13.000000
50%          36.000000
75%          83.000000
max       40235.000000
Name: time_spent_clean, dtype: float64


In [183]:
print((df['time_spent_clean']/60).describe().round(2))

count    247940.00
mean          1.40
std           3.60
min           0.00
25%           0.22
50%           0.60
75%           1.38
max         670.58
Name: time_spent_clean, dtype: float64


In [184]:
(df['time_spent_clean'] / 60 > 15).value_counts()

time_spent_clean
False    245134
True       2806
Name: count, dtype: int64

In [185]:
# converting time spent to minutes
df['time_spent_min'] = df['time_spent_clean']/60

In [186]:
# Count of outliers per step ( above 15min per step)
outliers = df[df['time_spent_min'] > 15].groupby('process_step')['time_spent_min'].agg(['count', 'mean', 'max','min']).round(2)
print(outliers)

              count   mean     max    min
process_step                             
confirm         562  21.05  243.02  15.02
start           854  24.82  670.58  15.02
step_1          310  24.54  171.43  15.12
step_2          248  21.13  362.72  15.03
step_3          832  20.74  111.53  15.02


In [187]:
# Drop anything above 15 minutes per step
df_15 = df[df['time_spent_min'] <= 15]

In [188]:
final_step_time = (df_15.groupby('process_step')['time_spent_clean']
                   .mean()
                   .reset_index())
final_step_time

,process_step,time_spent_clean
0,confirm,125.046627
1,start,48.802161
2,step_1,49.270185
3,step_2,84.864138
4,step_3,111.800990


In [189]:
# final aggregation
final_step_time = (df_15.groupby(['variation','process_step',])['time_spent_min']
                   .mean()
                   .reset_index()
                   .round(2)
                   .pivot(index='process_step', columns='variation', values='time_spent_min'))
final_step_time

variation,Control,Test
process_step,,
confirm,1.68,2.29
start,0.87,0.77
step_1,0.72,0.90
step_2,1.46,1.38
step_3,1.99,1.76


In [190]:
final_step_time = (df_15.groupby('variation')['time_spent_min']
                   .mean()
                   .reset_index()
                   .round(2))
final_step_time

,variation,time_spent_min
0,Control,1.19
1,Test,1.14


# KPI 3 : Error rate

In [191]:
df_sorted = df.sort_values(['visit_id', 'date_time']).reset_index(drop=True)

In [192]:
# 2. Map each step to a numeric rank
step_rank = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}
df_sorted['step_rank'] = df_sorted['process_step'].map(step_rank)

In [193]:
df_sorted.head(5)

,client_id,client_tenure_yr,client_tenure_month,client_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,variation,visitor_id,visit_id,process_step,date_time,next_time,time_spent_clean,time_spent_min,step_rank
0,3561384,4.0,56.0,60,Unknown,2,63130.44,6,9,Test,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,2017-04-26 13:23:09,52.0,0.866667,4
1,7338123,7.0,88.0,24,Male,2,26436.73,6,9,Test,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,2017-04-09 16:21:12,16.0,0.266667,0
2,7338123,7.0,88.0,24,Male,2,26436.73,6,9,Test,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,2017-04-09 16:21:21,9.0,0.150000,1
3,7338123,7.0,88.0,24,Male,2,26436.73,6,9,Test,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,2017-04-09 16:21:35,14.0,0.233333,2
4,7338123,7.0,88.0,24,Male,2,26436.73,6,9,Test,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:35,2017-04-09 16:21:41,6.0,0.100000,1


In [194]:
# 3. Get the previous step and its rank within each visit
df_sorted['prev_step'] = df_sorted.groupby('visit_id')['process_step'].shift(1)
df_sorted['prev_rank'] = df_sorted.groupby('visit_id')['step_rank'].shift(1)


In a funnel, going **backward** means you were at a higher step and dropped to a lower one :

- `prev_rank = 3`, `step_rank = 1` → user dropped back → `3 - 1 = 2` ✅
- `prev_rank = 1`, `step_rank = 3` → user moved **forward** → `1 - 3 = -2` → clamped to 0 ✅

So `prev_rank - step_rank` makes perfect sense here :
- **Positive result** = user went backward in the funnel
- **Negative result** = user moved forward (not a backward distance, so set to 0)
It does **not** remove the row — the row stays in your data.

Setting it to 0 just means : **"this user didn't go backward, so their backward distance is 0"**

Think of it like this :

| user | prev_rank | step_rank | backward_distance |
|------|-----------|-----------|-------------------|
| A | 3 | 1 | 2 (went backward) |
| B | 1 | 3 | **0** (moved forward, no backward distance) |
| C | 2 | 2 | **0** (stayed at same step) |

User B and C are still in your dataset, they just contribute **0** to any backward distance calculation — which makes sense, they didn't drop back.

So when you later do a sum or average of `backward_distance`, forward-moving users won't inflate the result, but they're still counted in your data.

In [195]:
# 4. Compute backward distance (weighted: step_3 -> step_1 = 2 errors)
df_sorted['backward_distance'] = df_sorted['prev_rank'] - df_sorted['step_rank']

In [196]:
df_sorted.loc[df_sorted['backward_distance'] < 0, 'backward_distance'] = 0
df_sorted['backward_distance'] = df_sorted['backward_distance'].fillna(0)

In [197]:
# A.Overall weighted error rate
overall_error_rate = df_sorted['backward_distance'].sum()*100 / len(df_sorted)
overall_error_rate

np.float64(11.561264822134387)

In [198]:
# B. Overall error rate per variation
error_by_variation = df_sorted.groupby('variation')['backward_distance'].mean()
error_by_variation = (error_by_variation * 100 ).round(2)
error_by_variation


variation
Control    11.13
Test       11.90
Name: backward_distance, dtype: float64

In [199]:
error_by_step = (
    df_sorted.groupby('prev_step')['backward_distance']
    .agg(total_errors='sum', error_rate='mean')
    .reindex(['start', 'step_1', 'step_2', 'step_3', 'confirm'])
)
error_by_step

,total_errors,error_rate
prev_step,,
start,0.0,0.000000
step_1,6375.0,0.113372
step_2,6939.0,0.141974
step_3,12388.0,0.995500
confirm,2963.0,1.276605


In [200]:
error_by_step_and_variation = (
    df_sorted.groupby(['variation', 'prev_step'])['backward_distance']
    .mean().round(2)
    .unstack('variation')
    .reindex(['start', 'step_1', 'step_2', 'step_3', 'confirm'])
)
error_by_step_and_variation

variation,Control,Test
prev_step,,
start,0.00,0.00
step_1,0.07,0.14
step_2,0.11,0.17
step_3,1.06,0.94
confirm,1.93,0.79


### Conclusion ###

The new design (test variation) performs better on Step 3 and the Confirm step, with a lower average backward navigation rate compared to the control variation. However, Steps 1 and 2 show a slightly higher average error rate in the test variation.


# KPI 4 : Return rate

**Definition**

the percentage of clients who needed more than one visit to complete the funnel

In [201]:
# Group by client and variation, count unique visit_ids per client
#  how many sessions each client had
sessions_per_client = (df.groupby(['client_id', 'variation'])['visit_id']
                       .nunique()
                       .reset_index())
sessions_per_client

,client_id,variation,visit_id
0,555,Test,1
1,647,Test,1
2,934,Test,1
3,1028,Control,1
4,1186,Control,1
...,...,...,...
46546,9999150,Test,1
46547,9999400,Test,1
46548,9999626,Test,1
46549,9999729,Test,3


In [202]:
# Flag clients who came back more than once
# True = returned, False = only visited once
sessions_per_client['is_return'] = sessions_per_client['visit_id'] > 1

In [203]:
(sessions_per_client.groupby('variation')['is_return'].mean() * 100).round(2)

variation
Control    18.11
Test       18.36
Name: is_return, dtype: float64

In [204]:
sessions_per_client.groupby('variation')['is_return'].sum()

variation
Control    3823
Test       4670
Name: is_return, dtype: int64

**Conclusion** : client  who saw the test variation needed more than 1 visit to complete the funnel which is (0,2%) more than the control variation